# Milestone 3: Testing and Validation Baseline
This notebook validates the initial prototype analyzing right-turn crash severity and speed limits. Run from the repository root or from its notebook location. The source CSV is not modified.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'data' / 'right_turn_crashes.csv').exists():
    root = root.parent
csv_path = root / 'data' / 'right_turn_crashes.csv'
assert csv_path.exists(), f'CSV not found: {csv_path}'
print('Repository:', root)
print('Dataset:', csv_path)

## 1. Load data and verify required columns

In [ ]:
df = pd.read_csv(csv_path)
required = {'CrashSeverity', 'SpeedLimit'}
missing = required - set(df.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'
print('Total rows:', len(df))
print('Missing values in required columns:')
print(df[['CrashSeverity', 'SpeedLimit']].isna().sum())
print('Duplicate rows (diagnostic only):', df.duplicated().sum())
display(df.head())

## 2. Clean only the variables used for this validation
Missing/blank severity, missing/non-numeric speed, and speed limits ≤ 0 are excluded **from these summaries only**. Duplicate rows are flagged but not automatically removed because two crashes may share identical recorded attributes.

In [ ]:
clean = df.copy()
clean['SpeedLimit'] = pd.to_numeric(clean['SpeedLimit'], errors='coerce')
clean['CrashSeverity'] = clean['CrashSeverity'].astype('string').str.strip()
valid = (clean['SpeedLimit'].notna() & (clean['SpeedLimit'] > 0)
         & clean['CrashSeverity'].notna() & clean['CrashSeverity'].ne(''))
clean = clean.loc[valid].copy()
print('Original rows:', len(df))
print('Valid rows:', len(clean))
print('Excluded rows:', len(df) - len(clean))
print('Retained percent:', round(100 * len(clean) / len(df), 2) if len(df) else 'N/A')
assert len(clean) > 0, 'No valid observations for analysis' 

## 3. Validate severity and speed-limit summaries

In [ ]:
severity_counts = clean['CrashSeverity'].value_counts().sort_index()
severity_by_speed = pd.crosstab(clean['SpeedLimit'], clean['CrashSeverity'])
display(severity_counts.rename('Count').to_frame())
display(severity_by_speed)
assert severity_counts.sum() == len(clean)
assert severity_by_speed.to_numpy().sum() == len(clean)
assert (severity_by_speed.sum(axis=0).sort_index() == severity_counts).all()
print('PASS: Both summaries account for all valid observations.')

## 4. Reproduce the prototype charts

In [ ]:
ax = severity_counts.plot(kind='bar', figsize=(8, 4))
ax.set(title='Distribution of Right-Turn Crash Severity', xlabel='Crash Severity', ylabel='Number of Crashes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
ax = severity_by_speed.plot(kind='bar', figsize=(10, 5))
ax.set(title='Crash Severity Outcomes by Speed Limit', xlabel='Speed Limit (mph)', ylabel='Number of Crashes')
plt.xticks(rotation=0)
plt.legend(title='Crash Severity', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Run reproducible automated tests
The test file `test_validation.py` is stored at the repository root. Tests use synthetic data with known answers.

In [ ]:
import subprocess
import sys
result = subprocess.run([sys.executable, '-m', 'pytest', '-v', 'test_validation.py'], cwd=root, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Automated tests failed'
print('PASS: Automated tests succeeded.')

## 6. Interpretation and limitations
- The tests establish consistency of the implemented summary calculations, not accuracy of source crash reports.
- Invalid or missing speed limits and severity values are excluded from these particular summaries; counts of exclusions are reported.
- Duplicate rows are reported, not removed, because duplicated attribute values do not necessarily indicate duplicated crashes.
- Crash frequencies are not crash rates: roadway exposure, traffic volumes, and confounding factors are not controlled.
- No causal interpretation of speed limit and crash severity is justified by these plots.

**After running:** Record the number of valid/excluded records and pytest results in README.md, then commit the notebook with its output.